# CLAHE
Contrast Limited Adaptive Histogram Equalization on the L channel of LAB. Enhancement baseline, not a haze model: answers "is local contrast stretching already enough?"

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/clahe" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same as the other method notebooks
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
CLIP_LIMIT = 2.0           # histogram clip, relative to a uniform histogram
TILE_GRID = (8, 8)         # number of tiles (x, y)

NAME = "clahe"

## Method

In [ ]:
def dehaze(bgr):
    #equalize only lightness (L), keep colour (a, b)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=TILE_GRID).apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2BGR)

## Preview a single image

In [ ]:
hazy = cv2.imread(hazy_paths[0])
out = dehaze(hazy)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, im, title in zip(ax, [hazy, out], ['hazy', NAME]):
    a.imshow(im[..., ::-1]); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = cv2.imread(p)
    t0 = time.time()
    out = dehaze(hazy)
    dt = time.time() - t0
    cv2.imwrite(f"{OUT}/{os.path.basename(p)}", out)
    print(f"{os.path.basename(p)} | {hazy.shape[1]}x{hazy.shape[0]} | {dt:.2f}s")

## Evaluate
Score the outputs with `dehazing/evaluation` (see `../DEHAZING_EVALUATION.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"clahe": OUT})
```